# TTM-Small-4M gốc — tiền kiểm Colab Free T4 (1024 → 1280)

**Không huấn luyện.** Notebook này chỉ tải mã theo commit SHA bất biến, chạy QA offline 22/22 và cho phép kiểm tra checkpoint T4 1024 bước ở chế độ CPU read-only. Không cung cấp chế độ execute hoặc khóa khoa học.

Nguồn PR #13, source commit `0b30376f4f4e377d277ba92ebb1dc82a07d2cf1c`. Bản checkpoint gốc trên Google Drive không được sửa, TEST 64 không được đọc. Cần phê duyệt science lock riêng trước bất kỳ TRAIN nào.


In [ ]:
# TTM Small4M original checkpoint continuation: source+CPU QA only
from pathlib import Path
import hashlib, subprocess, sys, urllib.request

REPO = 'minhtri22/TokenToModel-Lab'
SOURCE_COMMIT = '0b30376f4f4e377d277ba92ebb1dc82a07d2cf1c'
BASE_MAIN = 'b57b44e7287f842e81050c1faee337f636333dd0'
ROOT = Path('/content/ttm_small4m_original_1280_source')
if ROOT.exists(): raise RuntimeError('BOOTSTRAP_ALREADY_EXISTS_NO_OVERWRITE')
ROOT.mkdir(parents=True)
FILES = {
 'ttm_t4_original_1024_to1280.py': (SOURCE_COMMIT,'tools/pilot_t4/original_1024to1280_20261109/ttm_t4_original_1024_to1280.py','1f539aedd04d7097f2cb26b0220b14bd106c8b65','55cf8f9f686d911411ed3284ba6baf00b7a4ed1d7cc255e3f4fed1b9a2f5365c'),
 'test_ttm_t4_original_1024_to1280_offline.py': (SOURCE_COMMIT,'tools/pilot_t4/original_1024to1280_20261109/test_ttm_t4_original_1024_to1280_offline.py','e10ce2f0ce78d1d9aa0a7a7170735fefe79a825c','fad6d4860493209a285276dc12f1fd26d47ceb694b4aa8e1e64edbde456f7e28'),
 'p02.py': (BASE_MAIN,'src/p02/transformer_core.py','9762343e06cf800b50574c2871ae1ed58537fb18',None),
 't4_train_science_a2.py': (BASE_MAIN,'tools/pilot_t4/a2_continuous1024/t4_train_science_a2.py','76c4d0155cd0736919f41682f240b47ad3d7c428',None),
 't4_capacity_smoke.py': (BASE_MAIN,'tools/pilot_t4/integrated/t4_capacity_smoke.py','c9a6f256ba975228683ee7de2a1090e649110e97',None),
 't4_eval.py': (BASE_MAIN,'tools/pilot_t4/integrated/t4_eval.py','bef006bdff173fba6d8ba4aa0ef687a1732fd7c4',None),
}
for name,(commit,repo_path,blob,sha256) in FILES.items():
    url=f'https://raw.githubusercontent.com/{REPO}/{commit}/{repo_path}'
    data=urllib.request.urlopen(url,timeout=60).read()
    actual_blob=hashlib.sha1(b'blob '+str(len(data)).encode()+b'\0'+data).hexdigest()
    if actual_blob != blob: raise RuntimeError('GITHUB_GIT_BLOB_DRIFT:'+name)
    if sha256 and hashlib.sha256(data).hexdigest()!=sha256:
        raise RuntimeError('SOURCE_SHA256_DRIFT:'+name)
    (ROOT/name).write_bytes(data)
print('TTM_ORIGINAL_1280_BOOTSTRAP=PASS_EXACT_GITHUB')
qa=subprocess.run([sys.executable,str(ROOT/'test_ttm_t4_original_1024_to1280_offline.py')],cwd=str(ROOT),check=False)
if qa.returncode: raise RuntimeError('OFFLINE_QA_FAILED_NO_CLAIM')
print('TTM_ORIGINAL_1280_OFFLINE_QA=PASS_22_OF_22')
print('TTM_ORIGINAL_1280_TRAINING=NOT_AUTHORIZED_NOT_EXECUTED')


## Bước kiểm tra checkpoint nguồn (tùy chọn, chỉ đọc)

Chỉ điền đúng đường dẫn thực tế trên Drive rồi chạy. Không tạo `CLAIM`, không chạy `optimizer.step()`.


In [ ]:
# Optional: only inspect original checkpoint; do NOT train.
# Manually mount Drive first (e.g. from google.colab import drive; drive.mount('/content/drive')).
from pathlib import Path
import subprocess, sys
PARENT_DIR = ''     # absolute Drive path ending with checkpoints/step_1024
TRAIN_VAL_NPZ = ''  # exact SHA256 184f68b4...; TRAIN+VAL ONLY, no TEST
PACK_MANIFEST = ''  # exact SHA256 ef2e6e67... ; contains_test=false
OUTPUT_FUTURE = ''  # a NEW nonexisting folder on Drive, reserved for a later approved run
if not all([PARENT_DIR,TRAIN_VAL_NPZ,PACK_MANIFEST,OUTPUT_FUTURE]):
    print('TTM_ORIGINAL_1280_INSPECT=HOLD_PATHS_NOT_CONFIGURED_NO_CLAIM')
else:
    cmd=[sys.executable,str(ROOT/'ttm_t4_original_1024_to1280.py'),'--mode','inspect',
        '--parent-dir',PARENT_DIR,'--pack',TRAIN_VAL_NPZ,'--pack-manifest',PACK_MANIFEST,
        '--p02',str(ROOT/'p02.py'),'--trainer',str(ROOT/'t4_train_science_a2.py'),
        '--capacity',str(ROOT/'t4_capacity_smoke.py'),'--evaluator',str(ROOT/'t4_eval.py'),
        '--output',OUTPUT_FUTURE]
    result=subprocess.run(cmd,check=False)
    if result.returncode: raise RuntimeError('PARENT_CHECKPOINT_CPU_INSPECT_HOLD')
    print('TTM_ORIGINAL_1280_COLAB_INSPECT=PASS_READONLY_NO_GPU_TRAINING')
